In [1]:
import time
import json
import csv
import os
import random
import pandas as pd
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from bs4 import BeautifulSoup

In [ ]:
# Config
SEARCH_KEYWORD = "đồ ăn"
OUTPUT_FILE = "products_details.csv"
PROGRESS_FILE = "progress.json"
# FAILED_FILE = "failed.txt"
MAX_PAGES = 102   # số trang muốn crawl

In [ ]:
# Selenium Setup
options = webdriver.ChromeOptions()
options.add_argument("--start-maximized")
options.add_argument("--disable-blink-features=AutomationControlled")
options.add_experimental_option("excludeSwitches", ["enable-automation"])
options.add_experimental_option('useAutomationExtension', False)

browser = webdriver.Chrome(options=options)

browser.execute_cdp_cmd("Page.addScriptToEvaluateOnNewDocument", {
    "source": """
        Object.defineProperty(navigator, 'webdriver', {
            get: () => undefined
        })
    """
})

In [ ]:
def check_captcha(browser, max_wait=60):
    try:
        start = time.time()
        while True:
            try:
                captcha_element = browser.find_element(
                    By.CSS_SELECTOR, "iframe[src*='captcha'], div.captcha-container"
                )
                if captcha_element.is_displayed():
                    print(f"⚠️ CAPTCHA xuất hiện! Vui lòng nhập trong {max_wait} giây...")
                    time.sleep(2)
                    if time.time() - start > max_wait:
                        print("⏱️ Quá thời gian chờ captcha, skip sản phẩm này.")
                        return False
                else:
                    break
            except:
                break
        return True
    except:
        return True


In [ ]:
# Helper: Progress Save/Load
def load_progress():
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, "r", encoding="utf-8") as f:
            return json.load(f)
    return {"page": 1, "product": 0, "row_index": 0}

def save_progress(page, product, row_index):
    with open(PROGRESS_FILE, "w", encoding="utf-8") as f:
        json.dump({"page": page, "product": product, "row_index": row_index}, f)


In [ ]:
# Scroll Support
def scroll_to_bottom(browser, pause_time=1, max_scrolls=10):
    last_height = browser.execute_script("return document.body.scrollHeight")
    for _ in range(max_scrolls):
        browser.execute_script("window.scrollTo(0, document.body.scrollHeight);")
        time.sleep(pause_time)
        new_height = browser.execute_script("return document.body.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height

In [ ]:
# Crawl Product Details 
def get_product_details(browser, url, sold_value=None, max_retries=3):
    for attempt in range(max_retries):
        try:
            browser.get(url)
            time.sleep(random.uniform(2, 5))
            check_captcha(browser)

            WebDriverWait(browser, 15).until(
                EC.presence_of_element_located((By.CSS_SELECTOR, "h1"))
            )

            scroll_to_bottom(browser, pause_time=1.5, max_scrolls=20)
            soup = BeautifulSoup(browser.page_source, "html.parser")

            category = None
            categories = soup.select("li.breadcrumb_item")
            if len(categories) > 1:
                category = categories[1].get_text(strip=True)

            store_name = soup.select_one("div.seller-name-v2__top")
            store_name = store_name.get_text(strip=True) if store_name else None

            original_price = soup.select_one("span.pdp-v2-product-price-content-originalPrice-amount")
            original_price = original_price.get_text(strip=True) if original_price else None

            discount_percent = soup.select_one("span.pdp-v2-product-price-content-originalPrice-discount")
            discount_percent = discount_percent.get_text(strip=True) if discount_percent else None

            comment_count = soup.select_one("span.container-star-v2-count")
            comment_count = comment_count.get_text(strip=True) if comment_count else None

            rating = soup.select_one("span.container-star-v2-score")
            rating = rating.get_text(strip=True) if rating else None

            name = soup.select_one("div.pdp-product-title")
            name = name.get_text(strip=True) if name else None

            sold = sold_value  

            if not name:
                raise Exception("Trang chi tiết chưa load đủ dữ liệu")

            # Tính price từ original_price + discount
            price = None
            try:
                if original_price:
                    original_price_num = float(original_price.replace(".", "").replace("₫", "").strip())
                    if discount_percent:
                        disc = int(discount_percent.replace("%", "").replace("-", "").strip())
                        price = int(original_price_num * (100 - disc) / 100)
                    else:
                        price = int(original_price_num)
            except:
                price = original_price

            return {
                "category": category,
                "store_name": store_name,
                "original_price": original_price,
                "discount_percent": discount_percent,
                "price": price,
                "comment_count": comment_count,
                "rating": rating,
                "name": name,
                "sold": sold,
                "link": url
            }

        except Exception as e:
            print(f"❌ Lỗi khi lấy {url}, thử lại ({attempt+1}/{max_retries})... Lý do: {e}")
            time.sleep(random.uniform(3, 6))


    return None

In [ ]:
# Main Crawl
progress = load_progress()
start_page = progress["page"]
start_product = progress["product"]
row_index = progress["row_index"]

if not os.path.exists(PROGRESS_FILE):
    save_progress(start_page, start_product, row_index)
    print("📂 Tạo mới progress.json")

print(f"🔄 Tiếp tục từ trang {start_page}, sản phẩm {start_product}")

# mở trang search
search_url = f"https://www.lazada.vn/catalog/?q={SEARCH_KEYWORD}"
browser.get(search_url)
time.sleep(random.uniform(3, 6))
check_captcha(browser)


In [ ]:
# CSV Header
fieldnames = [
    "category", "store_name", "original_price",
    "discount_percent", "price", "comment_count", "rating",
    "name", "sold", "link"
]
if not os.path.exists(OUTPUT_FILE):
    with open(OUTPUT_FILE, "w", newline="", encoding="utf-8-sig") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()


In [ ]:
# Crawl từng trang
for page in range(start_page, MAX_PAGES + 1):
    print(f"📄 Đang crawl trang {page}...")
    browser.get(f"{search_url}&page={page}")
    time.sleep(random.uniform(3, 6))
    check_captcha(browser)
    scroll_to_bottom(browser, pause_time=1.5, max_scrolls=10)

    soup = BeautifulSoup(browser.page_source, "html.parser")
    products = soup.select(".Bm3ON")

    for idx, p in enumerate(products):
        if page == start_page and idx < start_product:
            continue  # skip sản phẩm đã crawl

        link_tag = p.select_one("a")
        if not link_tag:
            continue
        product_url = "https:" + link_tag["href"]

        sold_tag = p.select_one("span._1cEkb")
        sold_value = sold_tag.get_text(strip=True) if sold_tag else None

        details = get_product_details(browser, product_url, sold_value)
        if details:
            with open(OUTPUT_FILE, "a", newline="", encoding="utf-8-sig") as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                writer.writerow(details)
        else:
            # Ghi placeholder vào CSV
            placeholder = {col: None for col in fieldnames}
            placeholder["link"] = product_url
            with open(OUTPUT_FILE, "a", newline="", encoding="utf-8-sig") as f:
                writer = csv.DictWriter(f, fieldnames=fieldnames)
                writer.writerow(placeholder)

            # # Ghi vào failed.txt 
            # with open(FAILED_FILE, "a", encoding="utf-8") as f:
            #     f.write(f"{row_index}\t{product_url}\n")

        # update progress
        row_index += 1
        save_progress(page, idx + 1, row_index)

        time.sleep(random.uniform(1.5, 4))

print("✅ Hoàn thành crawl!")
browser.quit()